In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
from src import config

In [3]:
import hopsworks

# connect to the project
project = hopsworks.login(
    project=config.HOPSWORKS_PROJECT_NAME,
    api_key_value=config.HOPSWORKS_API_KEY,
    cert_folder="./hopsworks-certs",
)

# connect to feature store
feature_store = project.get_feature_store()

# connect to feature group 
feature_group = feature_store.get_feature_group(
    name=config.FEATURE_GROUP_NAME,
    version=config.FEATURE_GROUP_VERSION,
)

2026-06-24 11:17:33,118 INFO: Initializing external client
2026-06-24 11:17:33,119 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-24 11:17:35,404 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960


In [ ]:
feature_group.read()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (27.50s) 


,pickup_hour,rides,pickup_location_id
0,2025-10-13 06:00:00+00:00,27,249
1,2026-02-15 16:00:00+00:00,24,209
2,2026-02-01 06:00:00+00:00,0,172
3,2025-06-02 00:00:00+00:00,6,36
4,2025-12-01 01:00:00+00:00,7,137
...,...,...,...
3223723,2025-03-15 00:00:00+00:00,1,63
3223724,2026-04-04 01:00:00+00:00,0,109
3223725,2026-01-27 22:00:00+00:00,22,74
3223726,2026-03-01 06:00:00+00:00,10,7


In [5]:
# Create feature view if it doesn't exists yet
# This feature view uses only on feature group, so the query is trival
try:
    feature_store.create_feature_view(
        name=config.FEATURE_VIEW_NAME,
        version=config.FEATURE_VIEW_VERSION,
        query=feature_group.select_all(),
    )
except:
    print(f"Feature view already existed, Skip creation.")

# Get feature view
feature_view = feature_store.get_feature_view(
        name=config.FEATURE_VIEW_NAME,
        version=config.FEATURE_VIEW_VERSION,
    )


Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/fs/22624/fv/time_series_hourly_feature_view/version/1


In [6]:
ts_data, _ = feature_view.training_data(
    description="Time-series hourly taxi rides",
)

2026-06-24 11:19:54,068 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (27.75s) 


In [7]:
ts_data.sort_values(by=['pickup_location_id','pickup_hour'], inplace=True)
ts_data

,pickup_hour,rides,pickup_location_id
2998466,2025-01-01 00:00:00+00:00,0,1
2360684,2025-01-01 01:00:00+00:00,0,1
1967311,2025-01-01 02:00:00+00:00,0,1
1698964,2025-01-01 03:00:00+00:00,0,1
1951347,2025-01-01 04:00:00+00:00,0,1
...,...,...,...
486606,2026-06-24 03:00:00+00:00,0,265
508732,2026-06-24 04:00:00+00:00,1,265
456827,2026-06-24 05:00:00+00:00,1,265
489253,2026-06-24 06:00:00+00:00,0,265


In [8]:
from src.data import transform_ts_data_info_feature_and_target

features, targets = transform_ts_data_info_feature_and_target(
    ts_data,
    input_seq_len=24*29, # one month
    step_size=23,
)

features_and_target = features.copy()
features_and_target['target_rides_next_hour'] = targets

print(f'{features_and_target=}')

100%|██████████| 262/262 [01:33<00:00,  2.79it/s]


features_and_target=        rides_previous_696_hours  rides_previous_695_hours  \
0                            0.0                       0.0   
1                            0.0                       0.0   
2                            0.0                       0.0   
3                            2.0                       0.0   
4                            0.0                       1.0   
...                          ...                       ...   
132218                       1.0                       3.0   
132219                       2.0                       2.0   
132220                       3.0                       1.0   
132221                       4.0                       1.0   
132222                       3.0                       1.0   

        rides_previous_694_hours  rides_previous_693_hours  \
0                            0.0                       0.0   
1                            0.0                       0.0   
2                            0.0                 

In [15]:
print(features_and_target['pickup_hour'].dtype)
print(features_and_target[['pickup_hour']].head())

datetime64[ns, UTC]
                pickup_hour
0 2025-01-30 00:00:00+00:00
1 2025-01-30 23:00:00+00:00
2 2025-01-31 22:00:00+00:00
3 2025-02-01 21:00:00+00:00
4 2025-02-02 20:00:00+00:00


In [16]:
from datetime import date, timedelta
from pytz import timezone
import pandas as pd
from src.data_split import train_test_split

cutoff_date = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=28)

print(f'{cutoff_date=}')

X_train, y_train, X_test, y_test = train_test_split(
    features_and_target,
    cutoff_date,
    target_colum_name='target_rides_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')


cutoff_date=Timestamp('2026-05-27 08:33:30.502586+0000', tz='UTC')
X_train.shape=(124712, 698)
y_train.shape=(124712,)
X_test.shape=(7511, 698)
y_test.shape=(7511,)


In [22]:
import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    """
    Given a set of hyper-parameters, it trains a model and compute an average
    validation error based on a 'TimeSeriesSplit
    """
    # pick hyper-parameters
    hyperparams = {
        "metric": "mae",
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 100),
    }

    tss = TimeSeriesSplit(n_splits=4)
    scores = []

    for train_index, val_index in tss.split(X_train):

        X_train_ = X_train.iloc[train_index, :].copy()
        X_val_ = X_train.iloc[val_index, :].copy()
    
        y_train_ = y_train.iloc[train_index].copy()
        y_val_ = y_train.iloc[val_index].copy()
    
        pipeline = get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)
    
        y_pred = pipeline.predict(X_val_)
        mae = mean_absolute_error(y_val_, y_pred)
    
        scores.append(mae)

    return np.array(scores).mean()

In [23]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)

[I 2026-06-24 11:43:11,272] A new study created in memory with name: no-name-517b7452-d46a-4af9-834a-91f8bfb8f03a
[I 2026-06-24 11:43:43,064] Trial 0 finished with value: 4.441531481229396 and parameters: {'num_leaves': 46, 'feature_fraction': 0.6872931905635206, 'bagging_fraction': 0.8181992714121384, 'min_child_samples': 82}. Best is trial 0 with value: 4.441531481229396.
[I 2026-06-24 11:44:45,272] Trial 1 finished with value: 4.395198692247796 and parameters: {'num_leaves': 119, 'feature_fraction': 0.8659610391897836, 'bagging_fraction': 0.9850429902199951, 'min_child_samples': 73}. Best is trial 1 with value: 4.395198692247796.
[I 2026-06-24 11:46:08,249] Trial 2 finished with value: 4.38154493175211 and parameters: {'num_leaves': 256, 'feature_fraction': 0.6265156762269598, 'bagging_fraction': 0.5712498968929125, 'min_child_samples': 63}. Best is trial 2 with value: 4.38154493175211.
[I 2026-06-24 11:48:16,550] Trial 3 finished with value: 4.3429617643728875 and parameters: {'num

In [24]:
best_params = study.best_trial.params
print(f'{best_params=}')

best_params={'num_leaves': 126, 'feature_fraction': 0.9842365401343145, 'bagging_fraction': 0.5438434099401888, 'min_child_samples': 30}


In [25]:
pipeline = get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeaturesengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](699,)","['rides_previous_696_hours','rides_previous_695_hours', 'rides_previous_694_hours',...,'pickup_hour','pickup_location_id', 'average_rides_last_4_weeks']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,699
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...00165326BF7E0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [26]:
prediction = pipeline.predict(X_test)
test_mae = mean_absolute_error(y_test, prediction)
print(f'{test_mae=:.4f}')

test_mae=7.0740


In [30]:
from pathlib import Path

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

print(MODELS_DIR.resolve())


D:\My-Github\taxi_demand_predictor\notebooks\models


In [31]:
import joblib

joblib.dump(pipeline, MODELS_DIR / "model.pkl")

['models\\model.pkl']

In [32]:
from hsml.schema import Schema
from hsml.model_schema import ModelSchema

input_schema = Schema(X_train)
output_schema = Schema(y_train)
model_schema = ModelSchema(input_schema=input_schema, output_schema=output_schema)


In [34]:
model_registy = project.get_model_registry()

model = model_registy.sklearn.create_model(
    name="taxi_demand_predictor_next_hour",
    metrics={
        "test_mae": test_mae
    },
    description="LIGHTGBM Regressor with a bit of hyper-parameter tuning",
    input_example=X_train.sample(),
    model_schema=model_schema
)

model.save( MODELS_DIR / 'model.pkl')

  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'models\model.pkl' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading d:\My-Github\taxi_demand_predictor\notebooks\models\model.pkl: 0.000%|          | 0/1149722 elapsed<…

Uploading C:\Users\AHMEDS~1\AppData\Local\Temp\tmp7w7u994n\input_example.json: 0.000%|          | 0/4029 elaps…

Uploading C:\Users\AHMEDS~1\AppData\Local\Temp\tmp7w7u994n\model_schema.json: 0.000%|          | 0/63705 elaps…

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/34960/models/taxi_demand_predictor_next_hour/1


Model(name: 'taxi_demand_predictor_next_hour', version: 1)